In [1]:
from llm_client import LLMClient

llm_client = LLMClient()

OPENROUTER LLM CLIENT
Environment file: C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\.env
Environment exists: True
API key loaded: True
Primary model: nvidia/nemotron-3.5-super-vl:free
Fallback model: baidu/ernie-5.5-flash-preview:free
Temperature: 0.2
Timeout: 300s


In [2]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  21%|██        | 41/199 [00:00<00:00, 384.61it/s]

Loading weights:  40%|████      | 80/199 [00:00<00:00, 356.70it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 685.93it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  40%|████      | 42/105 [00:00<00:00, 377.00it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 825.32it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 722 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [3]:
# ============================================================
# BUILD REAL CONTEXT
# ============================================================

query = (
    "How does oxygen concentration affect "
    "flame behavior in microgravity?"
)


# Hybrid
hybrid_results = hybrid_retriever.search(
    query=query,
    dense_top_k=20,
    bm25_top_k=20,
    top_k=20
)


# Enrichment
enriched_results = metadata_filter.enrich_results(
    hybrid_results
)


# Reranking
reranked_results = reranker.rerank(
    query=query,
    candidates=enriched_results,
    top_k=2
)


# Selection
selected_results = context_selector.select(
    reranked_results
)


# Context
context = context_builder.build(
    selected_results
)


print("=" * 80)
print("CONTEXT READY")
print("=" * 80)

print(context)


CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...



RERANKING COMPLETE
Top K: 2

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 717
Reranker Score: -2.5601
Hybrid Rank: 2
Dense Rank: 1
BM25 Rank: 5

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 720
Reranker Score: -2.8376
Hybrid Rank: 1
Dense Rank: 2
BM25 Rank: 3
CONTEXT READY
[EVIDENCE 1]

Dataset summary: SUMMARY-DATASET (statistics over all 179 real ISS tests)

# Dataset overview — NASA PSI-159 CFI-G Normal Flames

## Summary

The knowledge base contains 179 real tests from the NASA PSI-159 CFI-G (Cool Flames
Investigation with Gases) experiment on the International Space Station (microgravity), run on
11 test days from 2021-03-29 to 2021-10-18.
Each test burns a small gaseous fuel jet (ethane, propane or butane, pure or diluted with
nitrogen) in an oxygen/nitrogen atmosphere.

- Oxygen: 18.0-40.0%. There are NO tests between 23.5% and 33% oxygen.
- Pressure: 51-304 kPa.
- Fuel flow: 0.0-3.215 mg/s.
- Outcomes: 132 of 17

In [4]:
# ============================================================
# TEST PROMPT BUILDER
# ============================================================

from llm.prompt_builder import PromptBuilder

prompt_builder = PromptBuilder()

prompt = prompt_builder.build(
    question=query,
    context=context,
    ml_prediction=None
)

print("=" * 80)
print("PROMPT BUILDER OUTPUT")
print("=" * 80)

print(prompt)

PROMPT BUILDER OUTPUT
You are the scientific reasoning component of the
"Flame in Freefall" microgravity combustion system.

The system retrieved scientific evidence from its knowledge
base: 179 REAL NASA ISS tests from PSI-159 CFI-G (Cool Flames
Investigation with Gases, normal gaseous diffusion flames).
Fuels: ethane, propane and butane (pure or diluted with nitrogen,
e.g. "30% butane"). Oxygen 18-40%, pressure 50-304 kPa.
Each test records: hot_ignition (did a hot flame light),
burn_time_s, end_mode (self_extinguished = went out by itself,
flow_terminated = still burning when the crew cut the fuel,
none = no ignition), peak_burner_temp_c, cool_flame and a
hazard index (0-100).

Your job is to produce a structured scientific response.

USER QUESTION

How does oxygen concentration affect flame behavior in microgravity?

RETRIEVED EXPERIMENTAL EVIDENCE

[EVIDENCE 1]

Dataset summary: SUMMARY-DATASET (statistics over all 179 real ISS tests)

# Dataset overview — NASA PSI-159 CFI-G Norma

In [5]:
# ============================================================
# TEST RESPONSE GENERATOR
# ============================================================

from llm.llm_client import LLMClient
from llm.response_generator import ResponseGenerator


llm_client = LLMClient()

response_generator = ResponseGenerator(
    llm_client=llm_client,
    prompt_builder=prompt_builder
)


response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


print("=" * 80)
print("FINAL STRUCTURED RESPONSE")
print("=" * 80)

print(response)

OPENROUTER LLM CLIENT
Environment file: C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\.env
Environment exists: True
API key loaded: True
Primary model: nvidia/nemotron-3.5-super-vl:free
Fallback model: baidu/ernie-5.5-flash-preview:free
Temperature: 0.2
Timeout: 300s

CALLING PRIMARY MODEL
Model: nvidia/nemotron-3.5-super-vl:free



Primary model succeeded.
LLM raw response type: <class 'dict'>
LLM raw response: {'answer': 'In these ISS microgravity tests, higher oxygen concentration was associated with more frequent ignition, longer burn times, and a greater likelihood that flames continued burning until the fuel flow was cut. At air-like oxygen (18.0–23.4%), 41 of 64 tests ignited and every lit flame self-extinguished (median burn time 8.0 s, longest 20.0 s). At enriched oxygen (33.2–40.0%), 91 of 115 tests ignited, 30 were still burning when fuel was cut, median burn time was 50.0 s, and the longest burn was 304.0 s. No tests exist between 23.5% and 33% oxygen, so the exact transition point is unknown. This is an association, not proven causation, because other test conditions also varied. The 33–40% oxygen levels are ISS test-chamber conditions, not the approximately 21% oxygen of the ISS cabin atmosphere. No machine-learning prediction is available for this question. The retrieved evidence does not provide i

In [6]:
# ============================================================
# FINAL STRUCTURED RESPONSE
# ============================================================

import json


# ------------------------------------------------------------
# GENERATE RESPONSE
# ------------------------------------------------------------

response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


# ------------------------------------------------------------
# DISPLAY STRUCTURED OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("FINAL STRUCTURED LLM RESPONSE")
print("=" * 80)

print(
    json.dumps(
        response,
        indent=2,
        ensure_ascii=False
    )
)


CALLING PRIMARY MODEL
Model: nvidia/nemotron-3.5-super-vl:free



Primary model succeeded.
LLM raw response type: <class 'dict'>
LLM raw response: {'answer': 'In microgravity, oxygen concentration is associated with flame persistence: at air-like oxygen (18-23%), all lit flames self-extinguished quickly, while at enriched oxygen (33-40%), many flames burned longer and some continued until the fuel flow was cut. No tests exist between 23.5% and 33% oxygen, so the transition point is unknown.', 'experimental_evidence': [{'experiment_id': 'CFI-21106-G6', 'oxygen_concentration': None, 'burn_time_s': None, 'peak_burner_temp_c': None, 'source': 'SUMMARY-OXYGEN', 'finding': 'Named as a longest-burning test in the air-like oxygen group (18.0-23.4%); all lit flames in this group self-extinguished, median burn time 8.0 s, longest 20.0 s, mean hazard index 10.9/100.'}, {'experiment_id': 'CFI-21106-G7', 'oxygen_concentration': None, 'burn_time_s': None, 'peak_burner_temp_c': None, 'source': 'SUMMARY-OXYGEN', 'finding': 'Named as a longest-burning test in the ai

In [7]:
# ============================================================
# SAVE FINAL RESPONSE AS JSON
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# GENERATE RESPONSE
# ------------------------------------------------------------

response = response_generator.generate(
    question=query,
    context=context,
    ml_prediction=None
)


# ------------------------------------------------------------
# OUTPUT PATH
# ------------------------------------------------------------

OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR / "final_response.json"
)


# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        response,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# CONFIRM
# ------------------------------------------------------------

print("=" * 80)
print("RESPONSE SAVED")
print("=" * 80)

print(f"File: {OUTPUT_FILE.resolve()}")
print(f"Size: {OUTPUT_FILE.stat().st_size} bytes")


CALLING PRIMARY MODEL
Model: nvidia/nemotron-3.5-super-vl:free



Primary model succeeded.
LLM raw response type: <class 'dict'>
LLM raw response: {'answer': 'In microgravity ISS tests, oxygen-enriched atmospheres (33-40% O2) were associated with more frequent ignition, longer burn times (median 50.0 s versus 8.0 s), and flames that continued burning until the fuel flow was cut. At air-like oxygen (18-23%), every lit flame self-extinguished and none burned until fuel cutoff. The exact oxygen concentration where this behavior changes cannot be determined because no tests were performed between 23.5% and 33% oxygen.}', 'experimental_evidence': [{'experiment_id': 'CFI-21106-G6', 'oxygen_concentration': None, 'burn_time_s': None, 'peak_burner_temp_c': None, 'source': 'SUMMARY-OXYGEN', 'finding': 'One of the longest-burning tests in the air-like oxygen group (18.0-23.4%); all flames in this group self-extinguished, with group median burn time 8.0 s and longest 20.0 s.'}, {'experiment_id': 'CFI-21106-G7', 'oxygen_concentration': None, 'burn_time_s': None,